In [3]:
import torch

## More on Computational Graphs

Conceptually, autograd keeps a record of data (tensors) and all executed operations (along with the resulting new tensors) in a
directed acyclic graph (DAG) consisting of Function objects. In this DAG, leaves are the input tensors, roots are the output tensors
By tracing this graph from roots to leaves, you can automatically compute the gradients using the chain rule.

**In a forward pass, autograd does two things simultaneously:**

* run the requested operation to compute a resulting tensor
* maintain the operation's gradient function in the DAG.

**The backward pass kicks off when .backward() is called on the DAG root. autograd then:**

* computes the gradients from each .grad_fn,
* accumulates them in the respective tensor's .grad attribute

* using the chain rule propagates all the way to the leaf tensors.

**NOTE**
* DAGs are dynamic in PyTorch An important thing to note is that the graph is recreated from scratch; after each
.backward() call, autograd starts populating a new graph. This is exactly what allows you to use control flow statements in
your model; you can change the shape, size and operations at every iteration if needed.

# Find Derivative of x^2 using AutoGrad

#### Forward Pass:
x -> sqrt(x) its gives y

In [4]:
x = torch.tensor(3.0, requires_grad=True)

In [5]:
y = x**2

In [6]:
x

tensor(3., requires_grad=True)

In [7]:
y

tensor(9., grad_fn=<PowBackward0>)

#### Backward Pass
```
x -> sqrt(x) gives y
     <----------- goes backward to find dy/dx i.e. gradient
```

In [8]:
# going backward to find derivative (it just stores the derivative not returnes it)
y.backward()

In [9]:
# use .grad to get the derivate
x.grad

tensor(6.)

# Find derivate of z = sin(y), y = x^2

#### Forward Pass
x -> sqrt(x) -> y -> sin(y)

In [10]:
x = torch.rand((1,), requires_grad=True)

In [11]:
y = x**2

In [12]:
z = torch.sin(y)

In [13]:
x, y, z

(tensor([0.0686], requires_grad=True),
 tensor([0.0047], grad_fn=<PowBackward0>),
 tensor([0.0047], grad_fn=<SinBackward0>))

#### Backward Pass
```
x -> sqrt(x) -> y -> sin(y) -> z
         <----------------- go in backward direction and use change rule to find the gradient
         dz/dx = dz/dy * dy/dx
```

In [14]:
x.backward()

In [15]:
x.grad

tensor([1.])

In [16]:
y.grad

C:\Users\itspr\AppData\Local\Temp\ipykernel_4036\486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\build\aten\src\ATen/core/TensorBody.h:494.)
  y.grad


In [17]:
z.grad

C:\Users\itspr\AppData\Local\Temp\ipykernel_4036\4110045842.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\build\aten\src\ATen/core/TensorBody.h:494.)
  z.grad


We can find derivates of the intermediate leafs using the retain_grad() funciton:

In [18]:
y.retain_grad()

In [19]:
y.backward()

In [20]:
y.grad

tensor([1.])

## Trainig a very small model using Neural Network for predicting some value

### 1. Manually Applying Gradients

In [21]:
#inputs
x = torch.tensor(6.7)   # Input feature
y = torch.tensor(0.0)   # True label (binary)

w = torch.tensor(1.0)   # weight
b = torch.tensor(0.0)   # bias

In [22]:
# Binary Cross-Entropy loss for scalar
def binary_cross_entropy(prediction, target):
    epsilon = 1e-8
    prediction = torch.clamp(prediction, min=epsilon, max=1-epsilon)
    return -(target * torch.log(prediction) + (1-target) * torch.log(1-prediction))

In [23]:
# Forward Pass
z = w * x + b # linear transformation
y_pred = torch.sigmoid(z) # predicted probability

# compoute binary cross-entropy loss
loss = binary_cross_entropy(y_pred, y)

In [24]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x # dz/dw = X
dz_db = 1 # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [25]:
print(f'Manual Gradient of loss w.r.t weight (dw): {dL_dw}')
print(f'Manual Gradient of loss w.r.t bias (db): {dL_db}')

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


### 2. Using AutoGrad for gradients

In [26]:
x = torch.tensor(6.7, requires_grad=True)
y = torch.tensor(0.0, requires_grad=True)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [27]:
z = w*x + b

In [28]:
y_pred = torch.sigmoid(z)

In [29]:
loss = binary_cross_entropy(y_pred, y)

In [30]:
x, y, z, y_pred, loss

(tensor(6.7000, requires_grad=True),
 tensor(0., requires_grad=True),
 tensor(6.7000, grad_fn=<AddBackward0>),
 tensor(0.9988, grad_fn=<SigmoidBackward0>),
 tensor(6.7012, grad_fn=<NegBackward0>))

In [31]:
loss.backward()

In [32]:
w.grad, b.grad

(tensor(6.6918), tensor(0.9988))

In [38]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
x

tensor([1., 2., 3.], requires_grad=True)

In [41]:
y = (x**2).mean()

In [42]:
y.backward()

In [44]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

### clearning grad

In [56]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [75]:
# forward pass
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [76]:
# backward pass
y.backward()

In [77]:
# find gradient
x.grad

tensor(4.)

**NOTE:** If we run the forward pass and backward pass and find the gradinet the result will not the be the one we would be expecting as we do so the gradients accumulates (kind of add again and again) so the result is different than that what it should be. Thus we need to make the gradient 0 before doing so.
Let say when we need to use the loop to train the NN multiple times.

Use the following:
```python
x.grad.zero_()
```

In [78]:
# clear the gradinet
x.grad.zero_()

tensor(0.)

### How to disable the gradient tracking

After we done with training (mainly) we don't need the gradient tracking but we need the values and variable so we disable the gradeint tracking the following three methods:

1. **requires_grad_(False)**
2. **detact()**
3. **torch.no_grad()**

### Method 1

In [80]:
x = torch.tensor(34.0, requires_grad=True)
x

tensor(34., requires_grad=True)

In [99]:
# using requires_grad_(False) to diable gradient tracking
x.requires_grad_(False)

tensor(34.)

#### Difference between requires_grad() adn requires_grad_()
| Feature | requires_grad | requires_grad_() |
|---------|---------------|------------------|
| Type | Attribute (bool) | Method (in-place) |
| Purpose | Check if gradients are tracked | Enable/disable gradient tracking |
| Callable? | ❌ No | ✅ Yes |
| Modifies Tensor? | ❌ No | ✅ Yes (in-place) |
| Returns | Boolean | The same tensor |

In [100]:
y = x*3+4
y

tensor(106.)

In [101]:
try:
    y.backward()
except Exception as e:
    print(f'Error:\n\t{e}')

Error:
	element 0 of tensors does not require grad and does not have a grad_fn


### Method 2

In [ ]:
x = torch.tensor(34.0, requires_grad=True)
x

In [ ]:
# using detach() to diable gradient tracking by creating a deep copy tensor with do not inherits the gradient part by default 
z = x.detach()
z

tensor(34.)

#### *Diffence between detach() and detach_():*
* we can get the gradient tracking back with detach() but not with detach_() it a is permanent change.


| Feature | detach() | detach_() |
|---------|----------|-----------|
| Operation type | Out-of-place (returns new tensor) | In-place (modifies original tensor) |
| Affects original tensor? | No | Yes |
| Shares storage? | Yes | Yes |
| Gradient tracking | Removed for returned tensor only | Removed for original tensor |
| Safety | Safer (original tensor still tracks gradients) | Riskier (irreversible change) |

In [94]:
y = x*3+4
y

tensor(106.)

In [95]:
try:
    y.backward()
except Exception as e:
    print(f'Error:\n\t{e}')

Error:
	element 0 of tensors does not require grad and does not have a grad_fn


In [ ]:
x.requires_grad_(True)

tensor(34., requires_grad=True)

In [105]:
y = x*3+4
y

tensor(106., grad_fn=<AddBackward0>)

In [107]:
x.requires_grad

True

### Method 3

In [108]:
x = torch.tensor(34.0, requires_grad=True)
x

tensor(34., requires_grad=True)

In [112]:
# using torch.no_grad() block to disable the gradient tracking
with torch.no_grad():
    y = x*3+4
y

tensor(106.)

In [110]:
try:
    y.backward()
except Exception as e:
    print(f'Error:\n\t{e}')

Error:
	element 0 of tensors does not require grad and does not have a grad_fn
